# Metadaten Auslesen
In diesem Beispiel werden die Metadaten der Dateien zu einem gegebenen Pfad 
ausgelesen und in der Datenbank gespeichert.
Außerdem wird gezeigt, wie ein Update aussehen könnte 

## Importe

In [ ]:
from logging import getLogger, FileHandler
from tqdm.notebook import tqdm
from pickle import dump, load
from pprint import pprint
from pathlib import Path
from itertools import batched
from torch.cuda import get_device_properties
from meipi.indexing.db import Base, DBPic, pgEngine, DBMeta
from meipi.indexing.preprocess import get_DBMeta_from_file
from meipi.indexing import appconf

## Settings

In [ ]:
DATA_DIR = "/home/padmin/Development/projekte/meipi-indexing/data/"
DOC_ROOT = "/home/rslsync/folders/"
image_root=Path(DOC_ROOT+"Bilder/")
logfile = "log.txt"
logger = getLogger("meta")
logger.setLevel("DEBUG")
logger.addHandler(FileHandler(logfile))

In [ ]:
engine = pgEngine(appconf.db_conn_string)
Base.metadata.create_all(engine.engine)

In [ ]:
#%%script true
metalist = []
batches = list(batched([path.as_posix() for path in image_root.rglob("**/*") if not path.is_dir()], n=100))
for batch in tqdm(batches,total = len(batches)):
    metalist.extend([get_DBMeta_from_file(path,docroot=DOC_ROOT,pool = "LocalBilder")
                     for path in batch])
with open(DATA_DIR+"newmeta.pkl", "wb") as f:
    dump(metalist,f)

In [ ]:
with open(DATA_DIR+"newmeta.pkl", "rb") as f:
    metalist = load(f)

In [ ]:
%%script true
with engine.Session() as session:
    session.add_all(metalist)
    session.flush()
    session.commit()